### Setup & Imports

1. [Qwen2.5 Unsloth Fine-Tuning](https://colab.research.google.com/drive/1Kose-ucXO1IBaZq5BvbwWieuubP7hxvQ?usp=sharing#scrollTo=2eSvM9zX_2d3)
2. [Islamic QA Data](https://huggingface.co/datasets/Omar-youssef/islamic-qa-egyptian-arabic)

In [ ]:
# %%capture
# import os, re
# if "COLAB_" not in "".join(os.environ.keys()):
#     !pip install unsloth  # Do this in local & cloud setups
# else:
#     import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
#     xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
#     !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
#     !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
#     !pip install --no-deps --upgrade "torchao>=0.16.0"
# !pip install transformers==4.56.2
# !pip install --no-deps trl==0.22.2

# !pip install --upgrade transformers

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

HF_TOKEN = user_secrets.get_secret('HF_TOKEN')
WANDB_TOKEN = user_secrets.get_secret("WANDB_TOKEN")

!huggingface-cli login --token {HF_TOKEN}
!wandb login {WANDB_TOKEN}


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Load the Model

In [ ]:
model_id = 'unsloth/Qwen2.5-7B-Instruct'

model, tokenizer = FastLanguageModel.from_pretrained(model_id,
                                                     load_in_4bit=True,
                                                     max_seq_length = 2048,
                                                     dtype=None,
                                                     use_gradient_checkpointing='unsloth')

In [ ]:
# model

#### We now add LoRA adapters for parameter efficient finetuning - this allows us to only efficiently train 1% of all parameters.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"]
    ,
    lora_alpha=16,  # Best to choose alpha = rank or rank*2
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=3407,
    use_rslora=True,    # To Support Rank Stabilized LoRA
    loftq_config=None   # Add LoftQ
)

# model

### Dataset Preparation

In [ ]:
dataset = load_dataset("Omar-youssef/islamic-qa-egyptian-arabic")

dataset

In [ ]:
dataset = dataset['train'].train_test_split(test_size=0.1, seed=3407)
train_ds = dataset['train']
eval_ds = dataset['test']

In [ ]:
# @title To Detect the Intent from the Source Topics (aqeedah, fiqh, history, general)

aqeedah_keywords = [
    "الله",
    "التوحيد",
    "العقيدة",
    "الإيمان",
    "الايمان",
    "الإحسان",
    "الاحسان",
    "الشريعة",
    "القدر",
    "الملائكة",
    "الملائكه",
    "الكتب السماوية",
    "الكتب السماويه",
    "الرسل",
    "الأنبياء",
    "الانبياء",
    "اليوم الآخر",
    "اليوم الاخر",
    "الحساب",
    "الجنة",
    "الجنه",
    "النار",
]

fiqh_keywords = [
    "الفقه",
    "الفقة",
    "التشريع",
    "أحكام",
    "احكام",
    "أركان",
    "اركان",
    "المذاهب",
    "الإجماع",
    "الاجماع",
    "السنة",
    "السنه",
    "الجماعة",
    "الجماعه",
    "بدعة",
    "بدعه",
    "عبادة",
    "عباده",
    "الصلاة",
    "الصلاه",
    "الزكاة",
    "الزكاه",
    "الشهادة",
    "الشهاده",
    "الصيام",
    "الحج",
    "الوضوء",
    "الطهارة",
    "الطهاره",
    "حديث",
    "قرآن",
    "قرأن",
]

history_keywords = [
    "التاريخ",
    "السيرة",
    "السيره",
    "سيرة",
    "سيره",
    "الصحابة",
    "الصحابه",
    "الخلفاء الراشدين",
    "أتباع",
    "اتباع",
    "الحاكم",
    "عصر",
    "غزوة",
    "غزوه",
    "معركة",
    "معركه",
    "حرب",
    "الردة",
    "الرده",
    "استشهاد",
    "تضحية",
    "تضحيه",
    "اجتهاد",
]

In [ ]:
def classify_intent(row):
  intent = ''

  # get the topic
  topic = row.get('source_topics', '')

  # clean it
  topic = str(topic).strip()

  if any(keyword in topic for keyword in aqeedah_keywords):
    intent = 'aqeedah'
  elif any(keyword in topic for keyword in fiqh_keywords):
    intent='fiqh'
  elif any(keyword in topic for keyword in history_keywords):
    intent='history'
  else:
    intent='general'

  return {'intent': intent}

In [ ]:
#@title Map this Function to dataset and save it as a Chcekpoint

train_ds = train_ds.map(classify_intent)

train_ds

In [ ]:
train_ds[0]

In [ ]:
dataset_id = "A7med-Ame3/islamic-qa-egyptian"
train_ds.push_to_hub(dataset_id, private=True)

### Train the Qwen3 Model

In [ ]:
train_ds = "A7med-Ame3/islamic-qa-egyptian"

train_ds = load_dataset(dataset_id)

train_ds

In [ ]:
eval_ds

In [ ]:
def formatting_prompt(examples):
    texts = []
    for question, answer in zip(examples['question'], examples['answer']):
        messages = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return texts

In [ ]:
args = SFTConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_steps = 5,
    num_train_epochs = 2,
    learning_rate = 2e-4,
    fp16 = True,
    logging_steps = 1,
    optim = "adamw_8bit",
    save_strategy = "steps",
    save_steps = 50,             # Every 50 Steps -> Save a Checkpoint
    eval_strategy = "steps",
    eval_steps = 50,
    weight_decay = 0.001,
    lr_scheduler_type = "linear",
    seed = 3407,
    # report_to = "wandb",
    padding_free = False,      # # Set to True if > 17 GB VRAM,
    output_dir = 'qwen3-tuned-islamic-qa'
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds['train'],
    eval_dataset = eval_ds,
    formatting_func = formatting_prompt,
    args = args
)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
# False if it is in the First Time 
# & Set it to True If the training is interrupted at any time and you run trainer.train(resume_from_checkpoint=True) again, 
# it will work normally and will resume from the last checkpoint that was saved!

# trainer = trainer.train(resume_from_checkpoint=False) # -> For First Time Running 

trainer = trainer.train(resume_from_checkpoint=True)  

In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

### Save the fine-tuned model

- After training completes (or if you stop it mid-way when you feel it’s sufficient), save the model.
- This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
# model.save_pretrained("lora_model")  # Local saving
# tokenizer.save_pretrained("lora_model")

# Online saving
model.push_to_hub("A7med-Ame3/qwen2.5_lora_model", token = HF_TOKEN)
tokenizer.push_to_hub("A7med-Ame3/qwen2.5_lora_model", token = HF_TOKEN) 

### Merge the Base Model with LoRA Adapter 

In [ ]:
# Push Merged to Hub
model.push_to_hub_merged(
    "A7med-Ame3/Qwen2.5-7B-LiveKit-16bit", 
    tokenizer, 
    save_method = "merged_16bit", 
    token = HF_TOKEN
)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "A7med-Ame3/qwen2.5_lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

### Inference

In [ ]:
import time

question = "ما هى اركان الاسلام ؟"

messages = [{
    'role': 'user',
    'content': question
}]

# prompt = "<|im_start|>" + question + "<|im_end|>" + "<|im_start|>" + "assistant\n"

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt = True, # assistant
    return_tensors='pt',
    return_dict=True,
    ).to(model.device)

start_time = time.time()

outputs = model.generate(**inputs,
                        max_new_tokens=128,
                        temperature=0.2,
                        do_sample=True)
end_time = time.time() - start_time

# outputs[0][inputs.shape[1]:] during decode ensures that the model prints only the answer 
# and removes the question text from the output.
output = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)

print(f"Inference Time = {end_time}")
print(output)

In [ ]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)
question = "ما الفرق بين النبي و الرسول ؟"

messages = [{
    'role': 'user',
    'content': question
}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be added for Generation
    return_tensors='pt'
).to(model.device)

_ = model.generate(
    input_ids = inputs,
    max_new_tokens=128,
    temperature=0.2,
    do_sample=True,
    streamer = TextStreamer(tokenizer, skip_prompt=True)
)

### Model Evaluation & Benchmarking

#### Measure Some Metrics

1. `TTFT (Time-To-First-Token)`: The time taken from the start of request processing until the model outputs the first token. (A very important metric for real-time streaming).

2. `Total Inference Time`: The total time to generate the complete answer. 

3. `Generated Tokens Count`: The number of new tokens generated by the model.

4. `TPS (Tokens Per Second)`: Generation speed, measured as: $$\text{TPS} = \frac{\text{Generated Tokens Count}}{\text{Total Inference Time}}$$

In [8]:
from transformers import TextStreamer

class LatencyStreamer(TextStreamer):
    def __init__(self, tokenizer):
        super().__init__(tokenizer, skip_prompt=True)
        self.start_time = None
        self.first_token_time = None
        self.generated_tokens = 0

    def put(self, value):
        current_time = time.time()
        # Record First Time when token generated
        if self.first_token_time is None and value.numel() > 0:
            self.first_token_time = current_time
        
        # Number of Generated Tokens
        self.generated_tokens += value.numel()
        super().put(value)

streamer = LatencyStreamer(tokenizer)

In [11]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "A7med-Ame3/qwen2.5_lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)

==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.14.1. vLLM: 0.26.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.7.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
        (layers): ModuleList(
          (0-1): 2 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
 

In [12]:
question = "ما الفرق بين النبي و الرسول ؟"

messages = [{
    'role': 'user',
    'content': question
}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be added for Generation
    return_tensors='pt'
).to(model.device)

prompt_length = inputs.shape[1]

start_time = time.time()
streamer.start_time = start_time
outputs = model.generate(
    input_ids = inputs,
    max_new_tokens=128,
    temperature=0.2,
    do_sample=True,
    streamer = TextStreamer(tokenizer, skip_prompt=True)
)
total_time = time.time() - start_time 

ttft = (streamer.first_token_time - start_time) if streamer.first_token_time else total_time
token_count = outputs[0].shape[0] - prompt_length
tps = token_count / total_time if total_time > 0 else 0

output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


النبي هو اللي بيدعو للتوحيد وبيبلغ رسالته، أما الرسول فبيكون قد أضاف له الوحي بالشريعة والتشريعات الجديدة أو المكملة.<|im_end|>


In [13]:
print("📊 INFERENCE BENCHMARK REPORT [Fine-Tuned Model]")
print("_"*40)
print(f"⏱️ Time-To-First-Token (TTFT) : {ttft:.4f} sec ({ttft*1000:.2f} ms)")
print(f"⏳ Total Inference Time        : {total_time:.4f} sec")
print(f"🔢 Total Output Tokens        : {token_count} tokens")
print(f"⚡ Tokens Per Second (TPS)     : {tps:.2f} tokens/sec")
print("_"*40)
print("\n📝 Output Answer:\n", output)

📊 INFERENCE BENCHMARK REPORT [Fine-Tuned Model]
________________________________________
⏱️ Time-To-First-Token (TTFT) : 3.3557 sec (3355.67 ms)
⏳ Total Inference Time        : 3.3557 sec
🔢 Total Output Tokens        : 47 tokens
⚡ Tokens Per Second (TPS)     : 14.01 tokens/sec
________________________________________

📝 Output Answer:
 النبي هو اللي بيدعو للتوحيد وبيبلغ رسالته، أما الرسول فبيكون قد أضاف له الوحي بالشريعة والتشريعات الجديدة أو المكملة.


In [3]:
model_id = 'unsloth/Qwen2.5-7B-Instruct'

base_model, tokenizer = FastLanguageModel.from_pretrained(model_id,
                                                     load_in_4bit=True,
                                                     max_seq_length = 2048,
                                                     dtype=None,
                                                     use_gradient_checkpointing='unsloth')

==((====))==  Unsloth 2026.7.5: Fast Qwen2 patching. Transformers: 5.14.1. vLLM: 0.26.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [9]:
import time

FastLanguageModel.for_inference(base_model)
question = "ما الفرق بين النبي و الرسول ؟"

messages = [{
    'role': 'user',
    'content': question
}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must be added for Generation
    return_tensors='pt'
).to(base_model.device)

prompt_length = inputs.shape[1]

start_time = time.time()
streamer.start_time = start_time
outputs = base_model.generate(
    input_ids = inputs,
    max_new_tokens=128,
    temperature=0.2,
    do_sample=True,
    streamer = TextStreamer(tokenizer, skip_prompt=True)
)
total_time = time.time() - start_time 

ttft = (streamer.first_token_time - start_time) if streamer.first_token_time else total_time
token_count = outputs[0].shape[0] - prompt_length
tps = token_count / total_time if total_time > 0 else 0

output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


في 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


الإسلام، يُستخدم المصطلحان "النبي" و"الرسول" للإشارة إلى الأشخاص الذين تلقوا الوحي من الله، لكنهما يشيران إلى فروق مهمة:

1. الرسول:
- الرسول هو شخص يتميز برسالة خاصة أو مهمتها.
- الرسالة التي يحملها الرسول هي عادة أكبر وأكثر أهمية من رسائل الأنبياء الآخرين.
- في القرآن الكريم، كلمة "رسول" تُستخدم لوصف سبعة أشخاص فقط: موسى، عيسى، محمد


In [10]:
print("📊 INFERENCE BENCHMARK REPORT [Base Model]")
print("_"*40)
print(f"⏱️ Time-To-First-Token (TTFT) : {ttft:.4f} sec ({ttft*1000:.2f} ms)")
print(f"⏳ Total Inference Time        : {total_time:.4f} sec")
print(f"🔢 Total Output Tokens        : {token_count} tokens")
print(f"⚡ Tokens Per Second (TPS)     : {tps:.2f} tokens/sec")
print("_"*40)
print("\n📝 Output Answer:\n", output)

📊 INFERENCE BENCHMARK REPORT [Base Model]
________________________________________
⏱️ Time-To-First-Token (TTFT) : 11.3830 sec (11383.05 ms)
⏳ Total Inference Time        : 11.3830 sec
🔢 Total Output Tokens        : 128 tokens
⚡ Tokens Per Second (TPS)     : 11.24 tokens/sec
________________________________________

📝 Output Answer:
 في الإسلام، يُستخدم المصطلحان "النبي" و"الرسول" للإشارة إلى الأشخاص الذين تلقوا الوحي من الله، لكنهما يشيران إلى فروق مهمة:

1. الرسول:
- الرسول هو شخص يتميز برسالة خاصة أو مهمتها.
- الرسالة التي يحملها الرسول هي عادة أكبر وأكثر أهمية من رسائل الأنبياء الآخرين.
- في القرآن الكريم، كلمة "رسول" تُستخدم لوصف سبعة أشخاص فقط: موسى، عيسى، محمد


### Final Inference Function

In [ ]:
from transformers import TextStreamer

def ask_sheikh(question, max_new_tokens=256, temperature=0.6, repetition_penalty=1.1, use_streamer=False):
    messages = [
        {
            'role': 'system',
            'content': (
                "أنت شيخ مصري حكيم وطيب، تتحدث بالعامية المصرية البسيطة والمحببة للقلب. "
                "ترد على الأسئلة الدينية بشكل مباشر وميسر ومبسط، وتستخدم عبارات طيبة ودعائية."
            )
        },
        {
            'role': 'user',
            'content': question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
        return_dict=True
    ).to(model.device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if use_streamer else None

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,  
        repetition_penalty=repetition_penalty,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        streamer=streamer
    )

    prompt_length = inputs['input_ids'].shape[1]
    output = tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True)
    
    return output.strip()


In [ ]:
question = "كم عدد الصلوات فى اليوم الواحد ؟ و لمازا نصوم رمضان ؟"

output = ask_sheikh(question, use_streamer=False)

print("الرد النهائي:")
print("_"*20)
print(output)

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [docs page](https://unsloth.ai/docs/basics/inference-and-deployment/saving-to-gguf)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb

- [Llama-Server & OpenAI Endpoint](https://unsloth.ai/docs/basics/inference-and-deployment/llama-server-and-openai-endpoint)

In [ ]:
model.save_pretrained_gguf(
    "qwen2_5_sheikh_gguf", 
    tokenizer, 
    quantization_method = "q4_k_m" 
)

In [ ]:
model.push_to_hub_gguf(
    "A7med-Ame3/Qwen2.5-7B-Sheikh-GGUF", 
    tokenizer, 
    quantization_method = "q4_k_m", 
    token = HF_TOKEN
)

### Llama Server & OpenAI Compatibility
- [Toutrial](https://unsloth.ai/docs/basics/inference-and-deployment/llama-server-and-openai-endpoint)

In [14]:
!apt-get update && apt-get install pciutils build-essential cmake curl libcurl4-openssl-dev -y

!git clone https://github.com/ggml-org/llama.cpp
!cmake llama.cpp -B llama.cpp/build -DBUILD_SHARED_LIBS=OFF -DGGML_CUDA=ON -DLLAMA_CURL=ON
!cmake --build llama.cpp/build --config Release -j --clean-first --target llama-server
!cp llama.cpp/build/bin/llama-server llama.cpp/

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [101 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,849 kB]
Get:12 https://cli.github.com/packages stable/main amd64 Packages [355 B]


In [15]:
# !pip install huggingface_hub hf_transfer
# import os
# os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

# from huggingface_hub import snapshot_download
# snapshot_download(
#     repo_id = "A7med-Ame3/Qwen2.5-7B-Sheikh-GGUF",
#     local_dir = "Devstral-2-123B-Instruct-2512-GGUF",
#     allow_patterns = ["*UD-Q2_K_XL*", "*mmproj-F16*"],
# )


import os
from huggingface_hub import hf_hub_download

# تنزيل ملف الـ GGUF المخصص بتاعك من حسابك
model_path = hf_hub_download(
    repo_id="A7med-Ame3/Qwen2.5-7B-Sheikh-GGUF",
    filename="unsloth.Q4_K_M.gguf", # اسم ملف الـ GGUF داخل الريبو
    local_dir="sheikh_model_dir"
)

print(f"Model Downloaded: {model_path}")

OSError: [Errno 28] No space left on device: '/kaggle/working/sheikh_model_dir/.cache/huggingface'

In [ ]:
import subprocess
import time

command = [
    "./llama.cpp/llama-server",
    "--model", "sheikh_model_dir/unsloth.Q4_K_M.gguf",
    "--alias", "sheikh-qwen",
    "--threads", "-1",
    "--n-gpu-layers", "999",
    "--prio", "3",
    "--min-p", "0.01",
    "--ctx-size", "4096",
    "--port", "8001",
    "--no-jinja"
]

with open("llama_server.log", "a") as log_file:
    llama_process = subprocess.Popen(command, stdout=log_file, stderr=log_file)

print(f"llama-server starting (PID): {llama_process.pid}")
time.sleep(20) 

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url = "http://127.0.0.1:8001/v1",
    api_key = "sk-no-key-required",
)

completion = client.chat.completions.create(
    model = "sheikh-qwen",
    messages = [
        {
            "role": "system", 
            "content": "أنت شيخ مصري حكيم وطيب، تتحدث بالعامية المصرية البسيطة وتساعد الناس وتدعيلهم."
        },
        {
            "role": "user", 
            "content": "كم عدد الصلوات في اليوم يا شيخنا؟"
        }
    ],
    temperature = 0.6
)

print(completion.choices[0].message.content)

### vLLM & OpenAI Compatibility

In [ ]:
# !pip install vllm -q

In [ ]:
# !nohup python3 -m vllm.entrypoints.openai.api_server \
#     --model "A7med-Ame3/Qwen2.5-7B-LiveKit-16bit" \
#     --port 8000 \
#     --gpu-memory-utilization 0.85 \
#     --max-model-len 4096 \
#     --dtype bfloat16 > vllm_server.log 2>&1 &

# --> OSError: Background processes not supported.

import subprocess

command = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "A7med-Ame3/Qwen2.5-7B-LiveKit-16bit",
    "--port", "8000",
    "--gpu-memory-utilization", "0.85",
    "--max-model-len", "4096",
    "--dtype", "bfloat16"
]

with open("vllm_server.log", "a") as log_file:
    process = subprocess.Popen(command, stdout=log_file, stderr=log_file)

print(f"vLLM Server Starting (PID): {process.pid}")

In [ ]:
!tail -n 30 vllm_server.log

In [ ]:
import time
from openai import OpenAI

time.sleep(15) 

client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="sk-no-key-required"
)

max_retries = 30
print("Connecting to vLLM Server")

for i in range(max_retries):
    try:
        response = client.chat.completions.create(
            model="A7med-Ame3/Qwen2.5-7B-LiveKit-16bit",
            messages=[
                {
                    "role": "system",
                    "content": "أنت شيخ مصري حكيم وطيب، تتحدث بالعامية المصرية البسيطة وتساعد الناس وتدعيلهم."
                },
                {
                    "role": "user",
                    "content": "كم عدد الصلوات في اليوم يا شيخنا؟"
                }
            ],
            temperature=0.6,
            max_tokens=256
        )
        print("\n✅Server is Available!\n")
        print("Response via vLLM:\n")
        print(response.choices[0].message.content)
        break
    except Exception as e:
        print(f"Try..Server is Loading({i+1}/{max_retries}) -Waiting 10 seconds..")
        time.sleep(10)